In [15]:
import json
import sqlite3
import requests
import pandas as pd

headers = {
    'sec-ch-ua': '"Chromium";v="112", "Google Chrome";v="112", "Not:A-Brand";v="99"',
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://triconresidential.com/',
    'DNT': '1',
    'sec-ch-ua-mobile': '?0',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36',
    'sec-ch-ua-platform': '"Windows"',
}

In [16]:
metro_location = input("Enter metro location (e.g. houston, dallas, atlanta): ").strip().lower()

records = []

In [ ]:
page = 1
last_page = 999

while page <= last_page:
    url = f'https://triconresidential.com/static/regions/{metro_location}/{page}.json'
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    results = response.json()
    last_page = results['meta']['last_page']

    for item in results['data']:
        availability = item.get('availability', {})
        records.append({
            'address': item.get('title', ''),
            'bed': item.get('beds', ''),
            'bath': item.get('baths', ''),
            'square_feet': item.get('square_feet', ''),
            'city': item.get('city', ''),
            'state': item.get('state', ''),
            'zip_code': item.get('zip', ''),
            'region_id': item.get('region_id', ''),
            'available': availability.get('display', ''),
            'price': item.get('rent', ''),
            'min_rent': item.get('min_rent', ''),
            'max_rent': item.get('max_rent', ''),
            'special': item.get('special', ''),
            'link': f"https://triconresidential.com/home/{item.get('slug', '')}",
            'self_tour_link': item.get('self_tour_url', ''),
            'virtual_tour_link': item.get('virtual_tour_url', ''),
            'unit_code': item.get('unit_code', ''),
            'metro_location': metro_location,
        })

    print(f'Page {page}/{last_page} complete — {len(results["data"])} listings')
    page += 1

print(f'\nTotal records: {len(records)}')

In [18]:
df = pd.DataFrame(records)
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 173 entries, 0 to 172
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   address            173 non-null    str    
 1   bed                173 non-null    int64  
 2   bath               173 non-null    float64
 3   square_feet        173 non-null    int64  
 4   city               173 non-null    str    
 5   state              173 non-null    str    
 6   zip_code           173 non-null    str    
 7   region_id          173 non-null    int64  
 8   available          173 non-null    str    
 9   price              173 non-null    int64  
 10  min_rent           173 non-null    int64  
 11  max_rent           173 non-null    int64  
 12  special            173 non-null    str    
 13  link               173 non-null    str    
 14  self_tour_link     173 non-null    str    
 15  virtual_tour_link  173 non-null    str    
 16  unit_code          173 non-null    in

,address,bed,bath,square_feet,city,state,zip_code,region_id,available,price,min_rent,max_rent,special,link,self_tour_link,virtual_tour_link,unit_code,metro_location
0,3254 Kelling Street Houston TX,4,2.0,1311,Houston,TX,77045,29646,Available,1549,1549,2014,Receive $500 Off First Full Month’s Rent,https://triconresidential.com/home/3254-kellin...,https://secure.triconresidential.com/public/lo...,https://www.insidemaps.com/app/walkthrough-tou...,90800518,houston
1,316 Summer Horse Drive La Marque TX,3,2.0,1414,La Marque,TX,77568-1237,29646,Available,1739,1739,2261,One Waived Application Fee ($55 Value),https://triconresidential.com/home/316-summer-...,https://secure.triconresidential.com/public/lo...,https://www.insidemaps.com/app/walkthrough-tou...,10800359,houston
2,9835 Willow Creek Commerce Dr 223 Tomball TX,4,2.5,2015,Tomball,TX,77375-2469,29646,Available,2559,2559,3327,One Month Rent Free + 50% Off Security Deposit,https://triconresidential.com/home/9835-willow...,https://secure.triconresidential.com/public/lo...,https://www.insidemaps.com/app/walkthrough-tou...,20800110,houston
3,9835 Willow Creek Commerce Dr 502 Tomball TX,3,2.5,1595,Tomball,TX,77375-2473,29646,Available,2409,2409,3132,One Month Rent Free + 50% Off Security Deposit,https://triconresidential.com/home/9835-willow...,https://secure.triconresidential.com/public/lo...,https://www.insidemaps.com/app/walkthrough-tou...,20800192,houston
4,9835 Willow Creek Commerce Dr 202 Tomball TX,3,2.5,1464,Tomball,TX,77375-2467,29646,Available,2079,2079,2703,One Month Rent Free + 50% Off Security Deposit,https://triconresidential.com/home/9835-willow...,https://secure.triconresidential.com/public/lo...,https://www.insidemaps.com/app/walkthrough-tou...,20800118,houston


In [19]:
dupes = df.duplicated(keep='first').sum()
print(f'Duplicate rows: {dupes}')
df = df.drop_duplicates(keep='first')
print(f'Rows after dedup: {len(df)}')

Duplicate rows: 0
Rows after dedup: 173


In [20]:
df.to_csv('tricon.csv', index=False)

In [21]:
conn = sqlite3.connect('tricon.db')
df.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(df)} rows to tricon.db -> listings table')
conn.close()

Wrote 173 rows to tricon.db -> listings table
